# Phase 2 Risk-to-Cost Retrieval Demo

This notebook presents the curated Phase 2 hybrid retrieval demo for the Enterprise Software & AI Compliance Analyzer. It imports the reusable demo module from `agent_brain.demo.curated_risk_to_cost` instead of duplicating query definitions or assertions.

The demo is aligned to `plans/query-scope.md` and demonstrates how local PostgreSQL vector retrieval and Neo4j graph traversal combine compliance evidence with renewal cost exposure.

## Prerequisites and run order

Before running this notebook, complete the runbook steps in `docs/setup-runbook.md`:

1. Start Docker services from the repository root.
2. Bootstrap PostgreSQL and pgvector in `database-layer/`.
3. Run `npm run reset:demo` and `npm run ingest`.
4. Validate `agent-brain/` with `python -m pytest`, `python -m ruff check src tests`, and `python -m mypy src`.
5. Run `python -m agent_brain.cli.project_graph` to project PostgreSQL records into Neo4j.
6. Optionally run `python -m agent_brain.cli.run_curated_demo` as a command-line smoke test.

The current embeddings are deterministic placeholder embedding vectors. They validate local retrieval plumbing and repeatability, but they are not production semantic embeddings.

## Environment assumptions

This notebook expects local services and environment values consistent with `.env.example`, `database-layer/.env.example`, and `agent-brain/.env.example`:

- PostgreSQL: `localhost:5432`
- Neo4j Bolt: `bolt://localhost:7687`
- PostgreSQL database: `compliance_analyzer`
- Embedding dimension: `8` while deterministic placeholder embeddings are in use

In [ ]:
from agent_brain.demo.curated_risk_to_cost import (
    CURATED_QUERIES,
    assert_curated_demo_passed,
    curated_result_rows,
    run_curated_demo,
)

len(CURATED_QUERIES)

## Curated query contract

The following query definitions are loaded from code and are intended to mirror `plans/query-scope.md`. Each query includes a natural-language prompt, a semantic phrase for vector retrieval, expected positive vendors, and the business purpose of the test.

In [ ]:
for query in CURATED_QUERIES:
    print(f"{query.query_id}: {query.title}")
    print(f"Prompt: {query.prompt}")
    print(f"Semantic phrase: {query.semantic_phrase}")
    print(f"Expected vendors: {', '.join(query.expected_vendor_names)}")
    print()

## Run hybrid retrieval demo

This cell runs all curated queries. Internally, the reusable module performs PostgreSQL vector retrieval, traverses Neo4j graph relationships, merges results into the expected risk-to-cost shape, calculates deterministic priority scores, and tracks whether expected positive vendors appeared.

In [ ]:
demo_results = run_curated_demo()
assert_curated_demo_passed(demo_results)

[(result.query.query_id, result.passed, result.matched_expected_vendor_names) for result in demo_results]

## Inspect result rows

The reusable `curated_result_rows()` function returns notebook-friendly dictionaries in the expected result shape from `plans/query-scope.md`: vendor, software, subscription, cost, renewal, risk, evidence, source document, and recommended review action.

In [ ]:
for result in demo_results:
    print(f"\n{result.query.query_id}: {result.query.title}")
    print(f"Matched expected vendors: {', '.join(result.matched_expected_vendor_names)}")
    for row in curated_result_rows(result):
        print(
            row['vendor_name'],
            row['software_name'],
            row['subscription_code'],
            row['annual_cost_usd'],
            row['risk_category'],
            row['risk_severity'],
            row['recommended_review_action'],
        )

## Explanation output

Each result also includes deterministic explanation text. This is intended for demo review and future agent grounding, not as a replacement for human compliance review.

In [ ]:
for result in demo_results:
    print(f"\n{result.query.query_id}: {result.query.title}")
    for explanation in result.explanations[:5]:
        print(f"- {explanation}")

## Limitations and reset instructions

- The notebook uses deterministic placeholder embedding vectors, not final semantic embeddings.
- If fixture data changes, rerun the database reset, ingestion, and graph projection steps from `docs/setup-runbook.md`.
- If expected positive vendors change, update `plans/query-scope.md`, the reusable curated demo module, tests, and this notebook together.
- The notebook provides retrieval evidence and prioritization context only. Cancellation or renewal recommendations still require future HITL workflow enforcement.